# 02 - Data Exploration

This notebook summarises the prepared NIR-SORT split and produces the figures used to describe the dataset. The numbers come from the prepared split CSV files. Train and validation counts are shown for model-development summaries; full-dataset totals are shown where they describe source coverage.

## Setup

This section loads the prepared split tables and defines the NIR representation used in the first prototype.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os

# Keep matplotlib caches outside the repo when the notebook is executed.
CACHE_ROOT = Path("/tmp/mdpp_v2_notebook_cache")
(CACHE_ROOT / "matplotlib").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_ROOT / "matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)


def resolve_mdpp_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == "mdpp_v2" and (candidate / "data_splits").exists():
            return candidate
        nested = candidate / "mdpp_v2"
        if (nested / "data_splits").exists():
            return nested
    raise RuntimeError("Could not find mdpp_v2 from the current working directory.")


MDPP_ROOT = resolve_mdpp_root()
SPLITS_ROOT = MDPP_ROOT / "data_splits"
SPLITS = ["train", "val", "test"]
VISIBLE_SPLITS = ["train", "val"]
SPLIT_LABELS = {"train": "Train", "val": "Validation", "test": "Test"}

# N/A and N/R both cover every fabric; absorbance is used for the current NIR representation.
USED_NIR = {
    "modality": "Benchtop_N",
    "instrument": "N",
    "data_format": "A",
    "short_code": "N/A",
    "description": "Benchtop NIR absorbance",
}



In [ ]:
sample_tables = []
measurement_tables = []
for split in SPLITS:
    sample_tables.append(pd.read_csv(SPLITS_ROOT / split / "samples.csv").assign(split=split))
    measurement_tables.append(pd.read_csv(SPLITS_ROOT / split / "measurements_manifest.csv").assign(split=split))

samples = pd.concat(sample_tables, ignore_index=True)
measurements = pd.concat(measurement_tables, ignore_index=True)

print(f"Loaded {len(samples)} sample rows and {len(measurements)} measurement rows.")


## 1. Split Totals

These are fabric-sample counts. Each fabric appears in exactly one split.

In [ ]:
split_totals = samples.groupby("split")["Sample ID"].nunique().reindex(SPLITS).rename("fabric_samples")
total_fabric_samples = int(samples["Sample ID"].nunique())

split_total_table = split_totals.rename_axis("split").reset_index()
split_total_table["split_label"] = split_total_table["split"].map(SPLIT_LABELS)
visible_split_total_table = split_total_table[split_total_table["split"].isin(VISIBLE_SPLITS)].copy()
visible_split_total_table = visible_split_total_table[["split_label", "fabric_samples"]]

print(f"Total fabric samples included in split: {total_fabric_samples}")


visible_split_total_table


## 2. Measurement Counts

These are measurement-file counts, not fabric counts. A single fabric can have several microscopy images and spectra records.

In [ ]:
modality_counts = (
    measurements.groupby(["split", "modality_group"])
    .size()
    .unstack(fill_value=0)
    .reindex(SPLITS)
)


modality_counts.loc[VISIBLE_SPLITS].rename(index=SPLIT_LABELS).rename(columns={"image": "microscopy/image files", "spectra": "spectra/NIR files"})


## 3. Spectroscopy Coverage

The spectroscopy records include benchtop and handheld devices, with absorbance and reflectance files. The benchtop `N/A` representation is used because it covers all 155 fabrics.

In [ ]:
spectra = measurements[measurements["modality_group"].eq("spectra")].copy()
spectra["nir_code"] = spectra["instrument"] + "/" + spectra["data_format"]
spectra["device_label"] = spectra["modality"] + " / " + spectra["instrument"]

device_distribution = (
    spectra.groupby(["modality", "instrument", "device_label"])
    .agg(files=("filename", "size"), fabrics=("sample_id", "nunique"))
    .reset_index()
    .sort_values(["modality", "instrument"])
)

device_format_distribution = (
    spectra.groupby(["modality", "instrument", "device_label", "data_format", "nir_code"])
    .agg(files=("filename", "size"), fabrics=("sample_id", "nunique"))
    .reset_index()
    .sort_values(["modality", "instrument", "data_format"])
)

spectra_breakdown = (
    spectra.groupby(["modality", "instrument", "data_format", "nir_code"])
    .agg(files=("filename", "size"), fabrics=("sample_id", "nunique"))
    .reset_index()
    .sort_values(["modality", "instrument", "data_format"])
)
spectra_breakdown["is_selected_nir"] = (
    spectra_breakdown["modality"].eq(USED_NIR["modality"])
    & spectra_breakdown["instrument"].eq(USED_NIR["instrument"])
    & spectra_breakdown["data_format"].eq(USED_NIR["data_format"])
)

spectra_by_split = (
    spectra.groupby(["split", "modality", "instrument", "data_format", "nir_code"])
    .agg(files=("filename", "size"), fabrics=("sample_id", "nunique"))
    .reset_index()
    .sort_values(["split", "modality", "instrument", "data_format"])
)

used_nir_rows = spectra[
    spectra["modality"].eq(USED_NIR["modality"])
    & spectra["instrument"].eq(USED_NIR["instrument"])
    & spectra["data_format"].eq(USED_NIR["data_format"])
]
used_nir_by_split = used_nir_rows.groupby("split").agg(files=("filename", "size"), fabrics=("sample_id", "nunique")).reindex(SPLITS)


print(f"Used NIR representation: {USED_NIR['modality']} / {USED_NIR['short_code']} ({USED_NIR['description']})")
print(f"Used NIR fabric coverage: {used_nir_rows['sample_id'].nunique()} fabrics")
print("Device coverage:")
for _, row in device_distribution.iterrows():
    print(f"- {row['device_label']}: {int(row['files'])} files, {int(row['fabrics'])} fabrics")


## 4. Spectroscopy Coverage Figure

The bars show fabric coverage per device/format combination. Labels also show the number of measurement files.

In [ ]:
device_format_plot = device_format_distribution.copy()
device_label_map = {
    "Benchtop_N": "Benchtop NIR",
    "Handheld_F": "Handheld F",
    "Handheld_S": "Handheld S",
}
device_format_plot["label"] = device_format_plot["modality"].map(device_label_map).fillna(device_format_plot["modality"]) + " " + device_format_plot["nir_code"]
device_format_plot = device_format_plot.sort_values(["fabrics", "files", "label"], ascending=True)

highlight_mask = (
    device_format_plot["modality"].eq(USED_NIR["modality"])
    & device_format_plot["instrument"].eq(USED_NIR["instrument"])
    & device_format_plot["data_format"].eq(USED_NIR["data_format"])
)
device_format_plot["is_selected"] = highlight_mask
colors = np.where(device_format_plot["is_selected"], "#2563eb", "#9ca3af")

fig, ax = plt.subplots(figsize=(9.5, 5.2))
bars = ax.barh(device_format_plot["label"], device_format_plot["fabrics"], color=colors)
ax.set_title("Spectroscopy device and format coverage", pad=14)
ax.set_xlabel("Fabric samples covered")
ax.set_ylabel("")
ax.set_xlim(0, max(178, int(device_format_plot["fabrics"].max()) + 23))
ax.grid(axis="x", alpha=0.18)
ax.set_axisbelow(True)

for bar, (_, row) in zip(bars, device_format_plot.iterrows()):
    label = f"{int(row['fabrics'])} fabrics, {int(row['files'])} files"
    if row["fabrics"] >= 45:
        x = bar.get_width() - 2.5
        ha = "right"
        color = "white"
    else:
        x = bar.get_width() + 2.5
        ha = "left"
        color = "#374151"
    ax.text(
        x,
        bar.get_y() + bar.get_height() / 2,
        label,
        ha=ha,
        va="center",
        fontsize=9,
        color=color,
    )

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color="#2563eb", label="Selected NIR representation"),
    plt.Rectangle((0, 0), 1, 1, color="#9ca3af", label="Other spectroscopy records"),
]
ax.legend(handles=legend_handles, loc="lower right", frameon=False, fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

print("Selected NIR representation is highlighted in blue.")


## 5. CNN Label Eligibility

The CNN classifier uses fabrics with one retained dominant-fibre class. Excluded fabrics remain available for composition-vector retrieval and fusion evaluation.

In [ ]:
eligible_mask = samples["cnn_eligible"].astype(bool)
exclusion_counts = samples["cnn_exclusion_reason"].fillna("ELIGIBLE").value_counts()
retained_classes = samples.loc[eligible_mask, "classification_label"].value_counts().sort_index()
too_small_classes = (
    samples.loc[samples["cnn_exclusion_reason"].eq("class_has_fewer_than_min_samples"), "dominant_fibre"]
    .value_counts()
    .sort_index()
)

print("CNN eligibility results")
print(f"CNN-eligible fabric samples total: {int(eligible_mask.sum())}")
print(f"CNN-excluded fabric samples total: {int((~eligible_mask).sum())}")
print(f"Excluded because no dominant fibre: {int(exclusion_counts.get('no_fibre_above_threshold', 0))}")
print(f"Excluded because tied dominant fibres: {int(exclusion_counts.get('tie', 0))}")
print(f"Excluded because class has fewer than 3 fabrics: {int(exclusion_counts.get('class_has_fewer_than_min_samples', 0))}")
print("CNN classes retained:", ", ".join(retained_classes.index.tolist()))
print("CNN classes excluded for being too small:", ", ".join(too_small_classes.index.tolist()))


eligibility_table = pd.DataFrame([
    {"quantity": "CNN-eligible fabric samples total", "value": int(eligible_mask.sum())},
    {"quantity": "CNN-excluded fabric samples total", "value": int((~eligible_mask).sum())},
    {"quantity": "Excluded because no dominant fibre", "value": int(exclusion_counts.get("no_fibre_above_threshold", 0))},
    {"quantity": "Excluded because tied dominant fibres", "value": int(exclusion_counts.get("tie", 0))},
    {"quantity": "Excluded because class has fewer than 3 fabrics", "value": int(exclusion_counts.get("class_has_fewer_than_min_samples", 0))},
])

eligibility_table


## 6. Training Class Distribution

This figure shows the CNN labels in the training split. The grey bar shows fabrics excluded from CNN class metrics.

In [ ]:
train_samples = samples[samples["split"].eq("train")].copy()
train_class_counts = train_samples["cnn_label"].value_counts().sort_values(ascending=True)
train_class_plot = train_class_counts.rename("fabric_samples").reset_index().rename(columns={"cnn_label": "label"})
train_class_plot["is_excluded"] = train_class_plot["label"].eq("EXCLUDED_CNN")
colors = np.where(train_class_plot["is_excluded"], "#9ca3af", "#2563eb")

fig, ax = plt.subplots(figsize=(9.5, 5.2))
bars = ax.barh(train_class_plot["label"], train_class_plot["fabric_samples"], color=colors)
ax.set_title("Training split CNN label distribution", pad=14)
ax.set_xlabel("Fabric samples")
ax.set_ylabel("")
ax.set_xlim(0, max(64, int(train_class_plot["fabric_samples"].max()) + 8))
ax.grid(axis="x", alpha=0.18)
ax.set_axisbelow(True)

for bar, (_, row) in zip(bars, train_class_plot.iterrows()):
    label = f"{int(row['fabric_samples'])}"
    if row["fabric_samples"] >= 18:
        x = bar.get_width() - 0.8
        ha = "right"
        color = "white"
    else:
        x = bar.get_width() + 0.8
        ha = "left"
        color = "#374151"
    ax.text(
        x,
        bar.get_y() + bar.get_height() / 2,
        label,
        ha=ha,
        va="center",
        fontsize=9,
        color=color,
    )

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color="#2563eb", label="Retained CNN classes"),
    plt.Rectangle((0, 0), 1, 1, color="#9ca3af", label="Excluded from CNN classifier"),
]
ax.legend(handles=legend_handles, loc="lower right", frameon=False, fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

print("Retained CNN classes are shown in blue; excluded fabrics are shown in grey.")


## 7. Result Summary

This summary collects the dataset, spectroscopy, and CNN-eligibility values used by later sections.

In [ ]:
from IPython.display import Markdown, display

split_summary = visible_split_total_table.rename(
    columns={"split_label": "split", "fabric_samples": "fabric samples"}
)

measurement_summary = (
    modality_counts.loc[VISIBLE_SPLITS]
    .rename(index=SPLIT_LABELS)
    .rename(columns={"image": "microscopy files", "spectra": "spectra files"})
    .reset_index()
    .rename(columns={"split": "split"})
)

spectroscopy_summary = spectra_breakdown.copy()
spectroscopy_summary["selected"] = spectroscopy_summary["is_selected_nir"].map({True: "yes", False: ""})
spectroscopy_summary = spectroscopy_summary[
    ["modality", "nir_code", "files", "fabrics", "selected"]
]

cnn_eligibility_summary = eligibility_table.copy()

retained_class_summary = (
    retained_classes.rename("fabric samples")
    .rename_axis("retained CNN class")
    .reset_index()
)

small_class_summary = (
    too_small_classes.rename("fabric samples")
    .rename_axis("excluded class")
    .reset_index()
)

summary_tables = {
    "Split totals": split_summary,
    "Measurement files": measurement_summary,
    "Spectroscopy representations": spectroscopy_summary,
    "CNN eligibility": cnn_eligibility_summary,
    "Retained CNN classes": retained_class_summary,
    "Classes excluded for size": small_class_summary,
}

for title, table in summary_tables.items():
    display(Markdown(f"### {title}"))
    display(table)
